# Phase 3 — Dataset Acquisition

> ⚠️ **Disclaimer:** PneumoniaMNIST is used strictly for **educational purposes**. This project is NOT a clinical diagnostic tool.

This notebook downloads the PneumoniaMNIST dataset from the official MedMNIST repository and validates it is ready for use. No manual image downloading required — the `medmnist` library handles everything.

### Step 0: Install Required Packages
Run this cell once to install all dependencies into your current Python environment.

In [ ]:
import subprocess, sys

packages = [
    'numpy',
    'matplotlib',
    'medmnist',
    'Pillow',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])

print('✅ All packages installed successfully.')

### Step 1: Download the Dataset
The `medmnist` library downloads a single compressed `.npz` file (~13MB) and caches it locally. It will NOT re-download on subsequent runs.

In [ ]:
import medmnist
from medmnist import INFO
import numpy as np

DATASET_NAME = 'pneumoniamnist'
DATA_CACHE_DIR = '../data/'

info = INFO[DATASET_NAME]
DataClass = getattr(medmnist, info['python_class'])

print(f'Dataset: {DATASET_NAME}')
print(f'Task:    {info["task"]}')
print(f'Labels:  {info["label"]}')
print(f'Channels:{info["n_channels"]}')
print()

print('Downloading training split...')
train_dataset = DataClass(split='train', download=True, root=DATA_CACHE_DIR)

print('Downloading validation split...')
val_dataset   = DataClass(split='val',   download=True, root=DATA_CACHE_DIR)

print('Downloading test split...')
test_dataset  = DataClass(split='test',  download=True, root=DATA_CACHE_DIR)

print()
print('✅ Dataset downloaded and cached successfully!')

### Step 2: Verify the Downloaded Data

In [ ]:
train_imgs   = train_dataset.imgs
train_labels = train_dataset.labels.squeeze()

val_imgs   = val_dataset.imgs
val_labels = val_dataset.labels.squeeze()

test_imgs   = test_dataset.imgs
test_labels = test_dataset.labels.squeeze()

print('=== Dataset Verification ===')
print(f'Train : {train_imgs.shape}  | Labels unique: {np.unique(train_labels)}')
print(f'Val   : {val_imgs.shape}    | Labels unique: {np.unique(val_labels)}')
print(f'Test  : {test_imgs.shape}   | Labels unique: {np.unique(test_labels)}')
print()
print(f'Image dtype : {train_imgs.dtype}')
print(f'Pixel range : [{train_imgs.min()}, {train_imgs.max()}]')
print(f'Image shape : {train_imgs[0].shape} (H x W)')

### Step 3: Preview Sample Images

In [ ]:
import matplotlib.pyplot as plt

label_names = {0: 'Normal', 1: 'Pneumonia'}

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
fig.suptitle('PneumoniaMNIST — Sample Images (28×28 Grayscale)', fontsize=13, fontweight='bold')

for cls in [0, 1]:
    indices = np.where(train_labels == cls)[0][:6]
    for j, idx in enumerate(indices):
        ax = axes[cls, j]
        ax.imshow(train_imgs[idx], cmap='gray')
        ax.axis('off')
        if j == 0:
            ax.set_title(f'Class {cls}: {label_names[cls]}', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

### Step 4: Class Distribution Check

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Class Distribution per Split', fontsize=13, fontweight='bold')

splits = [
    ('Train', train_labels),
    ('Val',   val_labels),
    ('Test',  test_labels),
]

for ax, (split_name, labels) in zip(axes, splits):
    counts = [np.sum(labels == c) for c in [0, 1]]
    ax.bar(['Normal (0)', 'Pneumonia (1)'], counts, color=['#4CAF50', '#F44336'])
    ax.set_title(f'{split_name} ({sum(counts)} total)')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts):
        ax.text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n✅ Phase 3 Complete — Dataset is verified and ready!')
print('→ Next: Open 01_eda_and_validation.ipynb for Phase 4 & 5')